In [21]:
import random, math, time, os

# Contains all .vrp file data in a single instance
class CVRPInstance:
    def __init__(self, name, capacity, coords, demands, depot_index):
        self.name = name # Instance name
        self.capacity = capacity # Vehivle capacity
        self.coords = coords # List of x,y coordinates for each node
        self.demands = demands # Demands for each node
        self.depot = depot_index # Index of each node (0-based)
        self.n = len(coords) # Number of nodes
        self.dist = self._compute_distance_matrix() # COmpute pairwise distances to save time for solver

    # Distance matrix, creates empty matrix and fills with all pairwise ditances (as integers)
    def _compute_distance_matrix(self):
        n = self.n
        dist = [[0]*n for _ in range(n)]
        for i in range(n):
            x1, y1 = self.coords[i]
            for j in range(i+1, n):
                x2, y2 = self.coords[j]
                d = int(round(math.hypot(x1 - x2, y1 - y2)))
                dist[i][j] = dist[j][i] = d
        return dist

# Parse all CVRP data
def parse_tsplib_cvrp(filename):
    name = None
    capacity = None
    coords = []
    demands = []
    depot_index = None

    section = None
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('COMMENT'):
                continue

            if ':' in line:
                key, val = [x.strip() for x in line.split(':', 1)]
                key = key.upper()
                if key == 'NAME':
                    name = val
                elif key == 'CAPACITY':
                    capacity = int(val)
                continue

            upper = line.upper()
            if upper == 'NODE_COORD_SECTION':
                section = 'COORD'
                continue
            elif upper == 'DEMAND_SECTION':
                section = 'DEMAND'
                continue
            elif upper == 'DEPOT_SECTION':
                section = 'DEPOT'
                continue
            elif upper == 'EOF':
                break

            if section == 'COORD':
                idx, x, y = line.split()
                coords.append((float(x), float(y)))
            elif section == 'DEMAND':
                idx, d = line.split()
                demands.append(int(d))
            elif section == 'DEPOT':
                idx = int(line)
                if idx == -1:
                    continue
                depot_index = idx - 1  # TSPLIB is 1-based index

    # Sanity Check: coordinates and demands should have same length
    assert len(coords) == len(demands), "coords and demands length mismatch"
    return CVRPInstance(name, capacity, coords, demands, depot_index)

In [22]:
# Sums each customer demand for a single route
def route_load(inst, route):
    return sum(inst.demands[i] for i in route)

# Calculate route cost
def route_cost(inst, route):
    d = inst.dist # Access distance matrix
    depot = inst.depot # Index of depot node
    if not route:
        return 0 # Empty route has 0 cost
        
    # Calculate cost from depot to first customer, then loop through consecuative customers, adding to final cost    
    cost = d[depot][route[0]]
    for i in range(len(route)-1):
        cost += d[route[i]][route[i+1]]
    cost += d[route[-1]][depot]
    return cost

# Calculate total cost of solution
def solution_cost(inst, sol):
    return sum(route_cost(inst, r) for r in sol)

# Check if route is feasible and all customers are visited exactly once
def is_feasible(inst, sol):
    cap = inst.capacity
    for r in sol:
        if route_load(inst, r) > cap:
            return False
    customers = set(range(inst.n))
    customers.remove(inst.depot) # Depot is not part of route
    seen = []
    for r in sol:
        seen.extend(r)
    return sorted(seen) == sorted(customers)

In [23]:
# Initial Cost using greedy heuristic
def greedy_initial_solution(inst):
    customers = [i for i in range(inst.n) if i != inst.depot] # List of all customers
    random.shuffle(customers) # Randomise customer order
    cap = inst.capacity
    sol = []
    current_route = []
    current_load = 0

    # interate through customers in random order
    for c in customers:
        d = inst.demands[c]
        if current_load + d <= cap: # If custoemr doesn't exceed capacity, add to route
            current_route.append(c)
            current_load += d
        else:
            sol.append(current_route) # If customer doesn't fit, close current route
            current_route = [c]
            current_load = d
    if current_route:
        sol.append(current_route)
    return sol

In [25]:
# N1 - Relocate Move
def neighbour_N1(inst, sol):
    sol = [r[:] for r in sol]
    if not sol:
        return sol

    # Pick non-empty source route
    non_empty = [i for i, r in enumerate(sol) if r]
    if not non_empty:
        return sol

    # Pick random customer inside route and rearrange
    src_idx = random.choice(non_empty)
    i_pos = random.randrange(len(sol[src_idx]))
    customer = sol[src_idx].pop(i_pos)
    if len(sol[src_idx]) == 0 and len(sol) > 1: # if route becomes empty, delete
        sol.pop(src_idx)

    # Choose target route (existing or new)
    if not sol or random.random() < 0.2:
        sol.append([])
        tgt_idx = len(sol) - 1
    else:
        tgt_idx = random.randrange(len(sol))

    # Insert customer in the target route
    insert_pos = random.randrange(len(sol[tgt_idx]) + 1)
    sol[tgt_idx].insert(insert_pos, customer)

    # Feasibility Check
    if not is_feasible(inst, sol):
        return sol  # Return anyway, SA will reject
    return sol

In [26]:
# N2 - Swap Move
def neighbour_N2(inst, sol):
    sol = [r[:] for r in sol]
    if len(sol) == 0:
        return sol

    # Collect all customer positions
    positions = []
    for r_idx, r in enumerate(sol):
        for p in range(len(r)):
            positions.append((r_idx, p))
    if len(positions) < 2: # Need at least 2 customers to swap
        return sol

    # Pick 2 random customers to swap
    (r1, p1), (r2, p2) = random.sample(positions, 2)
    sol[r1][p1], sol[r2][p2] = sol[r2][p2], sol[r1][p1]

    # Feasibility Check
    if not is_feasible(inst, sol):
        return sol
    return sol

In [27]:
def simulated_annealing(inst,
                        neighbourhood='N1',
                        T_start=1000.0, # Initial Temp
                        T_end=1e-3, # Stopping Temp
                        alpha=0.995, # Cooling Rate
                        max_iter=50000): # max iterations
    assert neighbourhood in ['N1', 'N2', 'N1N2'] # Validate neighbourhood chocie

    # Build initial solution, and store as current best
    current = greedy_initial_solution(inst)
    current_cost = solution_cost(inst, current)
    best = [r[:] for r in current]
    best_cost = current_cost

    # Initialise start temp and iterations
    T = T_start
    it = 0

    # SA Loop
    while T > T_end and it < max_iter:
        it += 1

        if neighbourhood == 'N1': 
            cand = neighbour_N1(inst, current)
        elif neighbourhood == 'N2':
            cand = neighbour_N2(inst, current)
        else:  # N1N2
            if random.random() < 0.5: # When using N1N2, randomly choose between both
                cand = neighbour_N1(inst, current)
            else:
                cand = neighbour_N2(inst, current)

        # Compute cost of neighbour
        cand_cost = solution_cost(inst, cand)
        delta = cand_cost - current_cost # Compute chaneg in cost

        # Acceptance rule
        if delta < 0 or random.random() < math.exp(-delta / T):
            current, current_cost = cand, cand_cost
            if current_cost < best_cost: # update global best
                best, best_cost = [r[:] for r in current], current_cost

        T *= alpha # Cool temp

    return best, best_cost

In [29]:
def run_all_instances(instances=["eil22","eil23","eil30","eil33"],
                      neighbourhoods=["N1","N2","N1N2"],
                      runs=5):

    results = []  # (instance, initial_cost, neighbourhood, best, avg, time)

    for name in instances:
        filename = f"{name}.vrp"
        inst = parse_tsplib_cvrp(filename)

        # compute initial greedy cost
        init_sol = greedy_initial_solution(inst)
        init_cost = solution_cost(inst, init_sol)

        print(f"\n=== {inst.name}  (n={inst.n}, capacity={inst.capacity}) ===")
        print(f"Initial greedy cost: {int(init_cost)}")

        for N in neighbourhoods:
            best_costs = []
            times = []

            for r in range(runs):
                start = time.time()
                _, best_cost = simulated_annealing(inst, neighbourhood=N)
                elapsed = time.time() - start

                best_costs.append(best_cost)
                times.append(elapsed)

            best_overall = min(best_costs)
            avg_cost = sum(best_costs)/runs
            avg_time = sum(times)/runs

            results.append((name, init_cost, N, best_overall, avg_cost, avg_time))

            print(f"{N:6s} | best = {int(best_overall):4d} | "
                  f"avg = {float(avg_cost):7.2f} | time = {float(avg_time):5.2f}s")

    return results

In [20]:
# Run the full experiment
results = run_all_instances()


=== eil22  (n=22, capacity=6000) ===
Initial greedy cost: 798
N1     | best =  281 | avg =  309.60 | time =  0.06s
N2     | best =  410 | avg =  446.60 | time =  0.06s
N1N2   | best =  302 | avg =  337.80 | time =  0.06s

=== eil23  (n=23, capacity=4500) ===
Initial greedy cost: 1450
N1     | best =  473 | avg =  512.60 | time =  0.06s
N2     | best =  623 | avg =  678.40 | time =  0.06s
N1N2   | best =  527 | avg =  574.60 | time =  0.06s

=== eil30  (n=30, capacity=4500) ===
Initial greedy cost: 1772
N1     | best =  395 | avg =  498.40 | time =  0.07s
N2     | best =  536 | avg =  644.60 | time =  0.07s
N1N2   | best =  496 | avg =  583.00 | time =  0.07s

=== eil33  (n=33, capacity=8000) ===
Initial greedy cost: 1785
N1     | best =  570 | avg =  639.80 | time =  0.06s
N2     | best =  908 | avg =  941.00 | time =  0.06s
N1N2   | best =  543 | avg =  660.00 | time =  0.07s
